# MLOps Group Project: NER Fine-tuning

This notebook completes the Named Entity Recognition (NER) fine-tuning workflow for the `mlops-group-project`.

We use the Hugging Face `open-ner-english` dataset and fine-tune `dslim/bert-base-NER` for token classification.


In [1]:
!pip install -q transformers torch accelerate datasets wandb huggingface_hub scikit-learn seaborn matplotlib seqeval evaluate


You should consider upgrading via the '/Users/anurag/code/mlops_project_group23/.venv/bin/python3 -m pip install --upgrade pip' command.


In [2]:
import os
import random
import numpy as np
import torch
import seaborn as sns
import matplotlib.pyplot as plt

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)
import wandb
import evaluate

sns.set(style='ticks', font_scale=1.2)

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

print(f'Using device: {device}')

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if device == 'cuda':
    torch.cuda.manual_seed_all(RANDOM_SEED)

WANDB_PROJECT = os.getenv('WANDB_PROJECT', 'mlops-group-project')
WANDB_RUN_NAME = os.getenv('WANDB_RUN_NAME', 'ner-fine-tune-run')
HF_REPO = os.getenv('HF_REPO', 'anuragvishwakarma02/mlops-group23-ner')
HF_TOKEN = os.getenv('HF_TOKEN')



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/Users/anurag/code/mlops_project_group23/.venv/lib/python3.9/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  

Using device: cpu


## 1. Dataset load and sanity checks

Load the English NER dataset and inspect its structure before tokenizing.


In [3]:
dataset = load_dataset('yongsun-yoon/open-ner-english')
print(dataset)
print('---')
print('Train features:')
print(dataset['train'].features)
print('---')
print('Sample train item:')
print(dataset['train'][0])


Generating validation split: 100%|██████████| 9178/9178 [00:00<00:00, 103492.06 examples/s]


DatasetDict({
    train: Dataset({
        features: ['text', 'entities'],
        num_rows: 36711
    })
    validation: Dataset({
        features: ['text', 'entities'],
        num_rows: 9178
    })
})
---
Train features:
{'text': Value('string'), 'entities': List({'entity_mentions': List(Value('string')), 'entity_type': Value('string')})}
---
Sample train item:
{'text': "---\nauthor:\n- |\n    Kelvin K. S. Wu, Ofer Lahav &\xa0Martin J. Rees\\\n    [Institute of Astronomy, Madingley Road, Cambridge CB3 0HA, UK]{}\nnocite:\n- '[@mesl90]'\n- '[@shect96]'\n- '[@be93; @be94; @smoot92; @benne96; @hanco97; @gs98; @bllw98; @lpt97; @treye98]'\ntitle: 'THE LARGE-SCALE SMOOTHNESS OF THE UNIVERSE'\n---\n\n\\#1[to 0pt[\\#1]{}]{}\n\n {#section .unnumbered}\n\n[**New measurements of galaxy clustering and background radiations provide improved constraints on the isotropy and homogeneity of the Universe on large scales. In particular, the angular distribution of radio sources and the X-Ray Backgrou

In [4]:
label_names = dataset['train'].features['ner_tags'].feature.names
print('Label names (first 20):', label_names[:20])
print('Total labels:', len(label_names))

model_name = 'dslim/bert-base-NER'
max_length = 128
batch_size = 16
num_train_epochs = 3
learning_rate = 5e-5

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
label_to_id = {label: i for i, label in enumerate(label_names)}
id_to_label = {i: label for label, i in label_to_id.items()}
print('Label to id mapping size:', len(label_to_id))


KeyError: 'ner_tags'

## 2. Tokenization and label alignment

Align token-level labels with tokenizer wordpieces so that only the first subtoken of a word contributes to the loss.


In [5]:
def align_labels(examples):
    tokenized_inputs = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True,
        max_length=max_length,
        padding='max_length',
    )

    all_labels = []
    for i, labels in enumerate(examples['ner_tags']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(labels[word_idx])
            else:
                label_ids.append(labels[word_idx] if labels[word_idx] != 0 else -100)
            previous_word_idx = word_idx
        all_labels.append(label_ids)

    tokenized_inputs['labels'] = all_labels
    return tokenized_inputs

prepared_dataset = dataset.map(
    align_labels,
    batched=True,
    remove_columns=dataset['train'].column_names,
)

print(prepared_dataset)
print('Example aligned labels (first example):', prepared_dataset['train'][0]['labels'][:20])


Map:   0%|          | 0/36711 [00:00<?, ? examples/s]


NameError: name 'tokenizer' is not defined

## 3. Model fine-tuning

Train a token classification model using `Trainer` and evaluate on the held-out split.


In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_names),
    id2label=id_to_label,
    label2id=label_to_id,
).to(device)

metric = evaluate.load('seqeval')

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []
    for prediction, label in zip(predictions, labels):
        valid_preds = []
        valid_labels = []
        for pred_id, label_id in zip(prediction, label):
            if label_id != -100:
                valid_preds.append(id_to_label[pred_id])
                valid_labels.append(id_to_label[label_id])
        true_predictions.append(valid_preds)
        true_labels.append(valid_labels)

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        'precision': results['overall_precision'],
        'recall': results['overall_recall'],
        'f1': results['overall_f1'],
        'accuracy': results['overall_accuracy'],
    }

# Initialize Weights & Biases if configured
wandb_enabled = False
if os.getenv('WANDB_API_KEY'):
    wandb.login(key=os.environ['WANDB_API_KEY'])
    wandb.init(
        project=WANDB_PROJECT,
        name=WANDB_RUN_NAME,
        config={
            'model_name': model_name,
            'batch_size': batch_size,
            'learning_rate': learning_rate,
            'max_length': max_length,
            'epochs': num_train_epochs,
            'dataset': 'yongsun-yoon/open-ner-english',
        },
    )
    wandb.watch(model, log='all', log_freq=50)
    wandb_enabled = True
else:
    print('WANDB_API_KEY not set; WandB logging disabled.')

training_args = TrainingArguments(
    output_dir='ner_results',
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=learning_rate,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_train_epochs,
    weight_decay=0.01,
    logging_steps=50,
    save_total_limit=2,
    seed=RANDOM_SEED,
    report_to='wandb' if wandb_enabled else 'none',
)

data_collator = DataCollatorForTokenClassification(tokenizer)
eval_split = 'validation' if 'validation' in prepared_dataset else 'test'
print('Using eval split:', eval_split)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=prepared_dataset['train'],
    eval_dataset=prepared_dataset[eval_split],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.save_model('ner_results/fine_tuned_model')
print('Model saved to ner_results/fine_tuned_model')
if HF_TOKEN:
    print(f'Pushing model and tokenizer to Hugging Face Hub repo: {HF_REPO}')
    trainer.push_to_hub(HF_REPO, use_auth_token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO, use_auth_token=HF_TOKEN)
else:
    print('HF_TOKEN not set; skipping Hugging Face Hub push.')


## 4. Evaluation and inference

Evaluate the fine-tuned model and run a quick inference example.


In [ ]:
results = trainer.evaluate()
print('Evaluation results:')
print(results)
if wandb_enabled:
    wandb.log(results)

from transformers import pipeline

ner_pipeline = pipeline(
    'ner',
    model='ner_results/fine_tuned_model',
    tokenizer=tokenizer,
    grouped_entities=True,
    device=0 if device == 'cuda' else -1,
)

sample_text = 'My name is Wolfgang and I live in Berlin. I work at Google and was born in 1990.'
print('Sample text:')
print(sample_text)
print('NER output:')
print(ner_pipeline(sample_text))
if wandb_enabled:
    wandb.finish()


## 5. Next steps

- Confirm the model evaluation metrics.
- Inspect the saved `ner_results/fine_tuned_model` directory.
- Use the `ner_pipeline` code block to run inference on new texts.
